# PREPROCESAMIENTO Y ETL CON PANDAS
## Elias Buitrago Bolivar
###

En el cuaderno anterior, 01 Pandas, se trabajó sobre una tabla que ya venía limpia. Aquí ocurre lo contrario: se parte de una base real de reservas de hotel y se la deja en condiciones de ser analizada.

En este cuaderno de jupyter se realiza un flujo de trabajo de extracción, transformación y carga. Los datos vienen sucios: filas repetidas, valores faltantes, categorías sin definir, fechas repartidas en tres columnas y valores imposibles. Al final se exporta una tabla limpia y verificable.

### Origen de los datos

Hotel Booking Demand, publicado en Kaggle por Jesse Mostipak, a partir del artículo de Antonio, de Almeida y Nunes.

Página del conjunto: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Archivo de descarga directa (no requiere credenciales de Kaggle): https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv

Son 119390 reservas de dos hoteles de Portugal entre 2015 y 2017. El archivo original trae 32 columnas y pesa unos 16 MB, así que la descarga tarda algunos segundos. En este cuaderno se cargan solo las 19 columnas que el análisis necesita.

### Importar librerías

In [1]:
import numpy as np
import pandas as pd

## EXTRAER

Leer el archivo tal como está, sin corregir nada todavía.

### Primero una muestra

El archivo pesa unos 16 MB y nadie carga entero un archivo que no conoce. Se traen mil filas para ver cómo viene la estructura y qué tan sucios están los datos. Si algo no cuadra, se corrige el llamado antes de descargar el resto.

In [2]:
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'

# nrows limita la lectura a las primeras filas
muestra = pd.read_csv(url, nrows=1000)
muestra.shape

(1000, 32)

In [3]:
# ¿Qué columnas trae y cómo se llaman?
muestra.columns.tolist()

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

In [4]:
muestra.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
# ¿Qué tipo asignó pandas a cada columna? La fecha llegó como texto
muestra.dtypes.head(10)

,0
hotel,object
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,object
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


In [6]:
# En mil filas ya se ven los problemas que tendrá el archivo completo
print(muestra.isna().sum().sum())
print(muestra.duplicated().sum())

1085
53


La muestra confirma tres cosas: son 32 columnas y el análisis no necesita todas, la fecha de llegada viene repartida en tres columnas de texto, y ya en mil filas hay faltantes. Con eso se decide cómo cargar el archivo completo.

### Ahora el archivo completo

In [7]:
# Se carga completo, con las 32 columnas.
# Para saber si una fila está repetida hay que comparar todas sus columnas,
# no solo las que se van a usar después
dataini = pd.read_csv(url)
dataini.shape

(119390, 32)

In [8]:
# Visualizar los primeros 5 registros incluyendo los encabezados
dataini.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [9]:
data = dataini.copy()
data.shape

(119390, 32)

### Duplicados: se revisan primero

El orden importa. Una fila duplicada se define comparando la fila completa, así que este paso va antes de descartar cualquier columna. Al revés se borrarían registros que en realidad son distintos.

In [10]:
# Cuántas filas están repetidas por completo, con las 32 columnas
data.duplicated().sum()

np.int64(31994)

In [11]:
# drop_duplicates conserva por defecto la primera aparición,
# según el orden en que las filas están hoy en la tabla
data = data.drop_duplicates()
data = data.reset_index(drop=True)
data.shape

(87396, 32)

### Seleccionar las columnas de trabajo

De las 32 columnas, este análisis usa 19. Traer de más obliga a leer una tabla más ancha en cada paso y no aporta nada.

In [12]:
columnas = ['hotel', 'is_canceled', 'lead_time',
            'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month',
            'stays_in_weekend_nights', 'stays_in_week_nights',
            'adults', 'children', 'babies',
            'meal', 'country', 'market_segment', 'distribution_channel',
            'agent', 'company', 'adr', 'reservation_status_date']

data = data[columnas]
data.shape

(87396, 19)

In [13]:
# El conteo de duplicados dejó de ser cero
data.duplicated().sum()

np.int64(3333)

Aquí está la razón por la que el orden importaba. Al quedarse con 19 de las 32 columnas, 3333 filas se volvieron indistinguibles entre sí. Si se hubiera hecho drop_duplicates después de recortar, se habrían borrado esos 3333 registros.

No podemos afirmar que sean reservas repetidas: el conjunto no tiene un identificador único de reserva, y en el archivo original esas filas se diferenciaban en alguna de las 13 columnas descartadas. Son indistinguibles con lo que conservamos, que no es lo mismo que ser duplicados. Por eso se mantienen.

In [14]:
data[data.duplicated(keep=False)].sort_values('lead_time').head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,agent,company,adr,reservation_status_date
37841,City Hotel,1,0,2016,February,13,1,1,2,0.0,0,BB,PRT,Online TA,TA/TO,9.0,NaN,106.0,2016-02-13
1921,Resort Hotel,0,0,2015,September,28,1,1,2,0.0,0,BB,FRA,Online TA,TA/TO,240.0,NaN,86.1,2015-09-30
29024,Resort Hotel,0,0,2017,April,3,1,0,1,0.0,0,BB,PRT,Direct,Direct,NaN,NaN,68.0,2017-04-04
28931,Resort Hotel,0,0,2017,March,31,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,NaN,521.0,65.0,2017-04-01
28930,Resort Hotel,0,0,2017,March,31,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,NaN,521.0,65.0,2017-04-01
28928,Resort Hotel,0,0,2017,March,31,0,1,1,0.0,0,BB,PRT,Corporate,Corporate,NaN,521.0,65.0,2017-04-01


### Diagnóstico

Instrucciones que dicen qué más está roto. Se ejecutan antes de escribir cualquier corrección.

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87396 entries, 0 to 87395
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   hotel                      87396 non-null  object 
 1   is_canceled                87396 non-null  int64  
 2   lead_time                  87396 non-null  int64  
 3   arrival_date_year          87396 non-null  int64  
 4   arrival_date_month         87396 non-null  object 
 5   arrival_date_day_of_month  87396 non-null  int64  
 6   stays_in_weekend_nights    87396 non-null  int64  
 7   stays_in_week_nights       87396 non-null  int64  
 8   adults                     87396 non-null  int64  
 9   children                   87392 non-null  float64
 10  babies                     87396 non-null  int64  
 11  meal                       87396 non-null  object 
 12  country                    86944 non-null  object 
 13  market_segment             87396 non-null  obj

In [16]:
# Cuántos faltantes tiene cada columna
faltantes = data.isna().sum()
faltantes[faltantes > 0]

,0
children,4
country,452
agent,12193
company,82137


In [17]:
# Cuántos valores distintos hay en cada columna
data.nunique().head(15)

,0
hotel,2
is_canceled,2
lead_time,479
arrival_date_year,3
arrival_date_month,12
arrival_date_day_of_month,31
stays_in_weekend_nights,17
stays_in_week_nights,35
adults,14
children,5


In [18]:
# Cómo están escritas realmente las categorías
data['meal'].value_counts(dropna=False)

,count
meal,
BB,67978
SC,9481
HB,9085
Undefined,492
FB,360


In [20]:
data['distribution_channel'].value_counts(dropna=False)

,count
distribution_channel,
TA/TO,69141
Direct,12988
Corporate,5081
GDS,181
Undefined,5


In [21]:
# Resumen numérico: aquí aparecen los valores imposibles
data[['adr', 'adults', 'children', 'babies', 'stays_in_week_nights']].describe()

,adr,adults,children,babies,stays_in_week_nights
count,87396.000000,87396.000000,87392.000000,87396.000000,87396.000000
mean,106.337246,1.875795,0.138640,0.010824,2.625395
std,55.013953,0.626500,0.455881,0.113597,2.053584
min,-6.380000,0.000000,0.000000,0.000000,0.000000
25%,72.000000,2.000000,0.000000,0.000000,1.000000
50%,98.100000,2.000000,0.000000,0.000000,2.000000
75%,134.000000,2.000000,0.000000,0.000000,4.000000
max,5400.000000,55.000000,10.000000,10.000000,50.000000


El diagnóstico deja una lista de tareas concreta:
pppp
* faltantes en children, country, agent y company
* la categoría Undefined en meal y en distribution_channel
* adr con un valor negativo y un máximo de 5400, muy lejos del resto
* reservas con cero adultos
* la fecha de llegada repartida en tres columnas y el mes en inglés

## TRANSFORMAR

### Valores faltantes

No hay una respuesta técnica correcta. Hay una decisión de negocio que debe quedar escrita.

In [22]:
# SUPUESTO DEL ANÁLISIS: interpretamos el vacío en children como ausencia de niños.
# El conjunto no permite demostrarlo, pero son solo 4 filas y la alternativa es descartarlas
data['children'] = data['children'].fillna(0)

In [23]:
# country: el país es desconocido, no es cero ni cadena vacía
data['country'] = data['country'].fillna('Desconocido')

In [24]:
# SUPUESTO DEL ANÁLISIS: interpretamos el vacío en agent como reserva sin agencia,
# y lo marcamos con el código 0 para poder distinguirlo después
data['agent'] = data['agent'].fillna(0)

In [25]:
# company: casi toda la columna está vacía, no aporta y se descarta
data['company'].isna().mean().round(4)

np.float64(0.9398)

In [26]:
data = data.drop(columns=['company'])
data.shape

(87396, 18)

In [27]:
# Verificación: no debe quedar ningún faltante
data.isna().sum().sum()

np.int64(0)

### Tipos de dato

In [28]:
# children y agent llegaron como decimales por culpa de los faltantes
data[['children', 'agent']].dtypes

,0
children,float64
agent,float64


In [29]:
data['children'] = data['children'].astype(int)
data['agent'] = data['agent'].astype(int)
data[['children', 'agent']].dtypes

,0
children,int64
agent,int64


In [ ]:
# is_canceled es una bandera de 0 y 1, no una medida:
# su promedio no es un promedio, es la proporción de reservas canceladas
data['is_canceled'].mean().round(4)

### Normalizar el texto

Un espacio al final o una mayúscula distinta parten un groupby en dos grupos sin avisar.

In [ ]:
data['hotel'] = data['hotel'].str.strip()
data['country'] = data['country'].str.strip().str.upper()
data['country'].head()

In [ ]:
# Undefined no es una categoría real: es un dato que nunca se registró
data['meal'] = data['meal'].replace('Undefined', 'SC')
data['meal'].value_counts()

In [ ]:
data['distribution_channel'] = data['distribution_channel'].replace('Undefined', 'Desconocido')
data['distribution_channel'].value_counts()

### Fechas

La fecha de llegada viene repartida en tres columnas y el mes está escrito en inglés.

In [ ]:
data[['arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month']].head()

In [ ]:
# El mes viene escrito en inglés. Se traduce a número con un diccionario:
# usar el formato %B haría depender el resultado de la configuración regional del equipo
meses = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
         'May': 5, 'June': 6, 'July': 7, 'August': 8,
         'September': 9, 'October': 10, 'November': 11, 'December': 12}

data['mes_llegada'] = data['arrival_date_month'].map(meses)
data['mes_llegada'].value_counts().sort_index()

In [ ]:
# Verificación: ningún mes quedó sin traducir
data['mes_llegada'].isna().sum()

In [ ]:
# pd.to_datetime arma la fecha a partir de tres columnas numéricas
data['fecha_llegada'] = pd.to_datetime({
    'year': data['arrival_date_year'],
    'month': data['mes_llegada'],
    'day': data['arrival_date_day_of_month'],
})
data['fecha_llegada'].head()

In [ ]:
data['reservation_status_date'] = pd.to_datetime(data['reservation_status_date'])
data['reservation_status_date'].dtype

In [ ]:
# El accesor .dt es a las fechas lo que .str es al texto
data['dia_semana_llegada'] = data['fecha_llegada'].dt.day_name()
data['trimestre'] = data['fecha_llegada'].dt.quarter
data[['fecha_llegada', 'mes_llegada', 'dia_semana_llegada', 'trimestre']].head()

In [ ]:
# Rango cubierto por los datos
print(data['fecha_llegada'].min())
print(data['fecha_llegada'].max())

### Valores imposibles

Un valor atípico puede ser un dato real o un error de captura. La diferencia se decide mirando el caso, no automáticamente.

In [ ]:
# Una tarifa diaria negativa no existe
data[data['adr'] < 0][['hotel', 'adr', 'adults', 'fecha_llegada']]

In [ ]:
# Y una tarifa de 5400 está muy lejos del resto
data['adr'].quantile([0.5, 0.9, 0.99, 1.0])

In [ ]:
data[data['adr'] > 1000][['hotel', 'adr', 'adults', 'market_segment']]

In [ ]:
# El valor negativo es incompatible con la definición de tarifa diaria y se descarta.
# El de 5400 es extremo, pero el conjunto no demuestra que sea un error:
# se fija una REGLA OPERATIVA de adr <= 1000 y se deja escrita.
# En un ETL real habría que validarlo contra la fuente antes de eliminarlo
data = data[(data['adr'] >= 0) & (data['adr'] <= 1000)]
data.shape

In [ ]:
# Una reserva sin ningún huésped tampoco tiene sentido
data['total_huespedes'] = data['adults'] + data['children'] + data['babies']
(data['total_huespedes'] == 0).sum()

In [ ]:
data = data[data['total_huespedes'] > 0]
data = data.reset_index(drop=True)
data.shape

### Columnas derivadas

In [ ]:
data['noches_totales'] = data['stays_in_weekend_nights'] + data['stays_in_week_nights']
data['noches_totales'].describe()

In [ ]:
# Hay reservas con cero noches: son entradas y salidas el mismo día
(data['noches_totales'] == 0).sum()

In [ ]:
# Este es el valor de la reserva, no el ingreso: incluye las reservas canceladas
data['valor_reserva'] = data['adr'] * data['noches_totales']
data['valor_reserva'].sum().round(2)

In [ ]:
# El ingreso efectivo solo cuenta lo que no se canceló.
# Una transformación técnicamente correcta puede producir un indicador
# semánticamente equivocado si se le pone el nombre que no es
data['ingreso_efectivo'] = data['valor_reserva'] * (1 - data['is_canceled'])
data['ingreso_efectivo'].sum().round(2)

In [ ]:
# La diferencia entre los dos es el valor que se perdió por cancelaciones
(data['valor_reserva'].sum() - data['ingreso_efectivo'].sum()).round(2)

In [ ]:
# np.where crea una columna condicional con dos salidas posibles
data['estado'] = np.where(data['is_canceled'] == 1, 'Cancelada', 'Efectiva')
data['estado'].value_counts()

In [ ]:
# pd.cut convierte una variable continua en bandas ordenadas
data['banda_estadia'] = pd.cut(data['noches_totales'],
                               bins=[-1, 1, 3, 7, 100],
                               labels=['express', 'corta', 'semana', 'larga'])
data['banda_estadia'].value_counts()

### Unir con un catálogo

El nombre completo del país no está en la tabla de reservas. Se trae desde un catálogo aparte con merge.

In [ ]:
paises = pd.DataFrame({
    'country': ['PRT', 'GBR', 'FRA', 'ESP', 'DEU', 'ITA', 'IRL', 'BEL', 'BRA', 'NLD'],
    'pais': ['Portugal', 'Reino Unido', 'Francia', 'España', 'Alemania',
             'Italia', 'Irlanda', 'Belgica', 'Brasil', 'Paises Bajos'],
    'continente': ['Europa', 'Europa', 'Europa', 'Europa', 'Europa',
                   'Europa', 'Europa', 'Europa', 'America', 'Europa'],
})
paises

In [ ]:
# validate detiene el proceso si la clave está repetida en el catálogo
# indicator muestra qué filas no encontraron pareja
data = data.merge(paises, on='country', how='left',
                  validate='many_to_one', indicator=True)
data['_merge'].value_counts()

In [ ]:
# Las reservas de países fuera del catálogo quedan sin nombre y se etiquetan
data['pais'] = data['pais'].fillna('Otro')
data['continente'] = data['continente'].fillna('Otro')
data = data.drop(columns=['_merge'])
data.shape

### Verificación de calidad

Un ETL no termina cuando el código corre sin error: termina cuando la tabla resiste estas preguntas.

In [ ]:
print('filas iniciales:', dataini.shape[0])
print('filas finales:', data.shape[0])
print('filas eliminadas:', dataini.shape[0] - data.shape[0])

In [ ]:
print('faltantes:', data.isna().sum().sum())
print('adr negativo:', (data['adr'] < 0).sum())
print('adr sobre el tope:', (data['adr'] > 1000).sum())
print('reservas sin huespedes:', (data['total_huespedes'] == 0).sum())
print('meses sin traducir:', data['mes_llegada'].isna().sum())

In [ ]:
# Las filas indistinguibles siguen ahí y es lo esperado: se documentó por qué se conservan
data.duplicated().sum()

In [ ]:
data.dtypes

## CARGAR

Guardar la tabla limpia donde la va a consumir el análisis.

In [ ]:
data.to_csv('reservas_hotel_limpio.csv', index=False)

In [ ]:
# utf-8-sig hace que Excel abra bien las tildes en Windows
data.to_csv('reservas_hotel_limpio_excel.csv', index=False, encoding='utf-8-sig')

### Comprobación sobre la tabla limpia

Con los datos ya procesados, las preguntas de negocio se responden en pocas líneas.

In [ ]:
kpi_hotel = (
    data
    .groupby('hotel', as_index=False)
    .agg(
        reservas=('hotel', 'size'),
        canceladas=('is_canceled', 'sum'),
        noches=('noches_totales', 'sum'),
        ingreso=('ingreso_efectivo', 'sum'),
        adr_promedio=('adr', 'mean'),
    )
)
kpi_hotel['tasa_cancelacion'] = (kpi_hotel['canceladas'] / kpi_hotel['reservas'] * 100).round(2)
kpi_hotel['adr_promedio'] = kpi_hotel['adr_promedio'].round(2)
kpi_hotel

In [ ]:
# Agrupar por un atributo que vino del catálogo
kpi_continente = (
    data
    .groupby('continente', as_index=False)
    .agg(reservas=('hotel', 'size'), ingreso=('ingreso_efectivo', 'sum'))
)
kpi_continente

In [ ]:
# Evolución mensual de las reservas efectivas
efectivas = data[data['estado'] == 'Efectiva']
efectivas.groupby('mes_llegada')['valor_reserva'].sum().round(2)

In [ ]:
# pivot_table pasa el resultado a formato ancho
tabla = data.pivot_table(index='hotel',
                         columns='estado',
                         values='valor_reserva',
                         aggfunc='sum',
                         fill_value=0)
tabla.round(2)